<a href="https://colab.research.google.com/github/fralfaro/inf449-portafolio/blob/main/tareas/tarea-02/homework2.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Tarea 2: Neural Networks Driven by Physics and CNNs for RGB Demosaicking

**Fotografia Computacional**  
**Universidad Tecnica Federico Santa Maria**

En esta tarea se estudia el problema de reconstruccion de imagenes RGB a partir de mediciones con mosaico Bayer obtenidas desde sensores reales. En particular, se busca comparar metodos clasicos de demosaicking con aproximaciones basadas en redes neuronales, incluyendo representaciones neuronales implicitas y redes convolucionales.


## 1. Enunciado

El objetivo general de la tarea es reconstruir una imagen RGB a partir de una medicion mosaico capturada por un dispositivo electronico, como un telefono, tablet o camara digital. La medicion corresponde a una imagen RAW o DNG generada por un sensor cubierto con un arreglo de filtros de color Bayer.

A partir de esta medicion, se deben implementar y analizar algoritmos de demosaicking basados en:

1. Metodos tradicionales de interpolacion.
2. Representaciones neuronales implicitas.
3. Redes neuronales convolucionales.

Los resultados deben compararse cuantitativa y cualitativamente con una imagen RGB de referencia, por ejemplo la imagen procesada automaticamente por el dispositivo.


## 2. Contexto teorico

Una camara digital usualmente no mide directamente los tres canales de color en cada pixel. En su lugar, el sensor registra una unica intensidad por posicion espacial, filtrada por un patron Bayer. Para un patron Bayer tipo **RGGB**, la distribucion local de filtros puede representarse como

$$
\begin{bmatrix}
R & G \\
G & B
\end{bmatrix}.
$$

Sea

$$
I \in \mathbb{R}^{H \times W \times 3}
$$

la imagen RGB ideal, donde $H$ y $W$ son la altura y el ancho de la imagen. La medicion mosaico puede modelarse como

$$
Y(i,j) = \sum_{c \in \{R,G,B\}} M_c(i,j) I_c(i,j),
$$

donde $M_c$ es una mascara binaria que indica si el canal $c$ fue medido en la posicion $(i,j)$.

El problema de **demosaicking** consiste en estimar una imagen RGB completa

$$
\hat{I} \in \mathbb{R}^{H \times W \times 3}
$$

a partir de la medicion incompleta $Y$.


## 3. Objetivos

Los objetivos especificos de esta tarea son:

1. Obtener o utilizar una imagen RAW/DNG junto con una imagen RGB de referencia.
2. Representar la medicion del sensor como un mosaico Bayer.
3. Reconstruir la imagen RGB usando un metodo clasico de demosaicking.
4. Reconstruir la imagen RGB usando una representacion neuronal implicita.
5. Reconstruir la imagen RGB usando una red neuronal convolucional.
6. Comparar los metodos mediante metricas objetivas y observaciones visuales.
7. Analizar el efecto de modificar la arquitectura o estrategia de entrenamiento de las redes.


## 4. Preparacion del entorno

En esta seccion se cargaran las librerias necesarias para trabajar con imagenes, arreglos numericos, visualizacion, metricas y entrenamiento de redes neuronales.

Las implementaciones finales deberian considerar, al menos:

- `numpy` para operaciones numericas.
- `matplotlib` para visualizacion.
- `Pillow` u `OpenCV` para lectura y procesamiento de imagenes.
- `torch` para definir y entrenar redes neuronales.
- `pandas` para organizar los resultados experimentales.


In [ ]:
# Dependencias para Google Colab
# En Colab, torch suele venir instalado. Esta celda asegura el resto de paquetes usados en homework2a.ipynb.
%pip install -q opencv-python scikit-image scipy pillow pandas matplotlib rawpy imageio

# Imports generales
from pathlib import Path
from collections import OrderedDict
import os
import time
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import cv2
import skimage
from skimage import data
from skimage.transform import resize, rescale
from skimage.metrics import peak_signal_noise_ratio
from skimage.metrics import structural_similarity as ssim
from scipy import io
from scipy.ndimage import correlate

try:
    import rawpy
except ImportError:
    rawpy = None

import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim.lr_scheduler as lr_scheduler
from torchvision.transforms import Resize, Compose, ToTensor, Normalize

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 5. Datos de trabajo

Para ordenar la tarea se usaran dos fuentes de datos:

1. **Experimento controlado:** la imagen `astronaut` de `skimage.data`, que ya se usa en `homework2a.ipynb`. Esta imagen permite construir un mosaico Bayer sintetico y comparar la reconstruccion contra una referencia RGB conocida.
2. **Experimento real:** los archivos `bursts_0006.dng`, `bursts_0006.tiff` y `bursts_0006.jpg` entregados en el repositorio del curso. Estos archivos permiten acercar el experimento al caso pedido en el enunciado: reconstruir desde una captura real.

En la primera etapa se trabajara con la imagen `astronaut`, porque facilita verificar que el pipeline completo funciona. Luego se puede reemplazar la medicion sintetica por la medicion RAW/DNG real.

La imagen RGB de referencia se denotara por

$$
I_{\text{ref}}.
$$

La medicion mosaico se denotara por

$$
Y.
$$


In [ ]:
# Descargar datos reales en Google Colab
# Estos archivos se usaran despues como extension del experimento controlado.
DATA_DIR = Path("/content") if Path("/content").exists() else Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://github.com/nelson10/ComputationalPhotography/raw/main/homework2"
FILES = [
    "bursts_0006.dng",
    "bursts_0006.tiff",
    "bursts_0006.jpg",
]

for filename in FILES:
    output_path = DATA_DIR / filename
    url = f"{BASE_URL}/{filename}"
    !wget -q -L -O "{output_path}" "{url}"

    size_kb = output_path.stat().st_size / 1024
    print(f"{output_path}: {size_kb:.1f} KB")

    if output_path.stat().st_size < 1024:
        print(f"Advertencia: {filename} parece no haberse descargado correctamente.")
        print("Primeros bytes del archivo:")
        !head -c 300 "{output_path}"
        print("\n")

DNG_PATH = DATA_DIR / "bursts_0006.dng"
TIFF_PATH = DATA_DIR / "bursts_0006.tiff"
JPG_PATH = DATA_DIR / "bursts_0006.jpg"


## 6. Experimento controlado con `skimage.data.astronaut`

Siguiendo el notebook base `homework2a.ipynb`, se usara la imagen `astronaut` como referencia RGB inicial. Esta eleccion tiene dos ventajas:

1. La imagen viene incluida en `scikit-image`, por lo que el experimento es reproducible en Colab.
2. Al conocer la imagen RGB original, se puede generar un mosaico Bayer sintetico y evaluar las reconstrucciones con metricas objetivas.

En este caso,

$$
I_{\text{ref}} = I_{\text{astronaut}}.
$$

A partir de esta imagen se construira el mosaico $Y$ usando el patron Bayer RGGB.

In [ ]:
# Carga de imagen controlada: astronaut
N = 64
L = 3

rgb_reference = data.astronaut()
rgb_reference = resize(rgb_reference, (N, N), anti_aliasing=True)
rgb_reference = rgb_reference.astype(np.float32)

# skimage puede entregar la imagen en [0, 1] despues de resize.
if rgb_reference.max() > 1:
    rgb_reference = rgb_reference / 255.0

plt.figure(figsize=(5, 5))
plt.imshow(rgb_reference)
plt.title("Referencia RGB: astronaut")
plt.axis("off");

## 7. Modelo de mosaico Bayer

Para un patron Bayer RGGB, se definen tres mascaras binarias:

$$
M_R, M_G, M_B \in \{0,1\}^{H \times W}.
$$

Cada mascara indica las posiciones del sensor donde se mide cada canal. La medicion mosaico puede escribirse como

$$
Y = M_R \odot I_R + M_G \odot I_G + M_B \odot I_B,
$$

donde $\odot$ representa el producto elemento a elemento.

Esta seccion debe construir o leer la medicion $Y$ y visualizar el patron de muestreo utilizado.


In [ ]:
# Mascaras Bayer RGGB y medicion mosaico
def bayer_masks(height, width, pattern="RGGB"):
    """Construye mascaras binarias para un patron Bayer de 2x2."""
    patterns = {
        "RGGB": np.array([[0, 1], [1, 2]]),
        "BGGR": np.array([[2, 1], [1, 0]]),
        "GRBG": np.array([[1, 0], [2, 1]]),
        "GBRG": np.array([[1, 2], [0, 1]]),
    }
    if pattern not in patterns:
        raise ValueError(f"Patron Bayer no soportado: {pattern}")
    tile = patterns[pattern]
    masks = np.zeros((height, width, 3), dtype=np.float32)
    for row in range(2):
        for col in range(2):
            masks[row::2, col::2, tile[row, col]] = 1.0
    return masks

def make_bayer_mosaic(rgb_image, masks):
    return np.sum(rgb_image * masks, axis=-1).astype(np.float32)

bayer_pattern = "RGGB"
bayer = bayer_masks(*rgb_reference.shape[:2], pattern=bayer_pattern)
mosaic = make_bayer_mosaic(rgb_reference, bayer)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(bayer)
axes[0].set_title("Patron Bayer RGGB")
axes[1].imshow(mosaic, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Medicion mosaico Y")
for ax in axes:
    ax.axis("off")
plt.tight_layout()


def load_real_capture(dng_path, reference_path):
    """Carga un mosaico RAW y una referencia RGB, si los archivos son validos."""
    if rawpy is None:
        raise RuntimeError("rawpy no esta disponible; ejecuta primero la celda de dependencias.")
    with rawpy.imread(str(dng_path)) as raw:
        raw_mosaic = raw.raw_image_visible.astype(np.float32)
        black = float(np.mean(raw.black_level_per_channel))
        white = float(raw.white_level)
        raw_mosaic = np.clip((raw_mosaic - black) / max(white - black, 1.0), 0.0, 1.0)
        pattern_indices = np.asarray(raw.raw_pattern)
        color_desc = raw.color_desc.decode("ascii") if isinstance(raw.color_desc, bytes) else str(raw.color_desc)
        raw_pattern = "".join(color_desc[int(index)] for index in pattern_indices.ravel())
    reference = np.asarray(Image.open(reference_path).convert("RGB"), dtype=np.float32) / 255.0
    reference = resize(reference, raw_mosaic.shape, anti_aliasing=True).astype(np.float32)
    return raw_mosaic, reference, raw_pattern

real_capture = None
try:
    if DNG_PATH.exists() and JPG_PATH.exists() and DNG_PATH.stat().st_size > 1024:
        real_capture = load_real_capture(DNG_PATH, JPG_PATH)
        print(f"Captura real cargada. Patron detectado: {real_capture[2]}")
    else:
        print("Captura real omitida: el DNG/JPG no esta disponible o parece invalido.")
except Exception as error:
    print(f"Captura real omitida: {error}")



## 8. Metodo clasico de demosaicking

Como linea base, se debe reconstruir la imagen RGB usando un metodo tradicional. Un enfoque simple consiste en interpolar los valores faltantes de cada canal a partir de las muestras disponibles.

La reconstruccion clasica se denotara como

$$
\hat{I}_{\text{clasico}}.
$$

Esta reconstruccion servira como referencia inicial para evaluar si los metodos neuronales entregan mejoras cuantitativas o visuales.


In [ ]:
# Demosaicking clasico por interpolacion bilineal
def demosaic_bilinear(mosaic, masks):
    # Los kernels separables estiman los valores faltantes de cada canal.
    kernels = [
        np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32),
        np.array([[0, 1, 0], [1, 4, 1], [0, 1, 0]], dtype=np.float32),
        np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32),
    ]
    channels = []
    for channel, kernel in enumerate(kernels):
        observed = mosaic * masks[..., channel]
        estimate = correlate(observed, kernel, mode="mirror") / 4.0
        # Conserva exactamente las muestras que el sensor si observo.
        channels.append(np.where(masks[..., channel] > 0, observed, estimate))
    return np.clip(np.stack(channels, axis=-1), 0.0, 1.0).astype(np.float32)

I_hat_classic = demosaic_bilinear(mosaic, bayer)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb_reference)
axes[0].set_title("Referencia RGB")
axes[1].imshow(mosaic, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Mosaico Bayer")
axes[2].imshow(I_hat_classic)
axes[2].set_title("Demosaicking bilineal")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

if real_capture is not None:
    real_mosaic, real_reference, real_pattern = real_capture
    if real_pattern not in {"RGGB", "BGGR", "GRBG", "GBRG"}:
        print(f"Patron {real_pattern} no reconocido; se usara RGGB como respaldo.")
        real_pattern = "RGGB"
    real_masks = bayer_masks(*real_mosaic.shape, pattern=real_pattern)
    real_classic = demosaic_bilinear(real_mosaic, real_masks)
    opencv_codes = {
        "RGGB": cv2.COLOR_BayerRG2RGB,
        "BGGR": cv2.COLOR_BayerBG2RGB,
        "GRBG": cv2.COLOR_BayerGR2RGB,
        "GBRG": cv2.COLOR_BayerGB2RGB,
    }
    real_opencv = cv2.cvtColor(real_mosaic.astype(np.float32), opencv_codes[real_pattern])
    real_opencv = np.clip(real_opencv, 0.0, 1.0)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(real_reference)
    axes[0].set_title("Referencia JPG")
    axes[1].imshow(real_mosaic, cmap="gray")
    axes[1].set_title(f"RAW Bayer {real_pattern}")
    axes[2].imshow(real_classic)
    axes[2].set_title("Bilineal")
    axes[3].imshow(real_opencv)
    axes[3].set_title("OpenCV")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 9. Representacion neuronal implicita

Una representacion neuronal implicita modela una senal como una funcion continua parametrizada por una red neuronal. En este caso, se busca aprender una funcion

$$
f_\theta: \mathbb{R}^3 \rightarrow \mathbb{R},
$$

donde la entrada corresponde a una coordenada espacial y un indice de canal,

$$
(x, y, c),
$$

y la salida corresponde a una intensidad estimada.

El entrenamiento utiliza solamente las posiciones observadas en el mosaico Bayer. La funcion de perdida puede escribirse como

$$
\mathcal{L}(\theta) = \frac{1}{|\Omega|} \sum_{(i,j,c) \in \Omega}
\left( f_\theta(x_i, y_j, c) - Y(i,j) \right)^2,
$$

donde $\Omega$ es el conjunto de muestras disponibles en el sensor.

La reconstruccion obtenida se denotara por

$$
\hat{I}_{\text{INR}}.
$$


In [ ]:
# Representacion neuronal implicita tipo SIREN
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_0=30.0, is_first=False):
        super().__init__()
        self.omega_0 = omega_0
        self.linear = nn.Linear(in_features, out_features)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1 / in_features, 1 / in_features)
            else:
                bound = np.sqrt(6 / in_features) / omega_0
                self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

class Siren(nn.Module):
    def __init__(self, hidden_features=256, hidden_layers=3, out_features=3):
        super().__init__()
        layers = [SineLayer(2, hidden_features, is_first=True)]
        layers += [SineLayer(hidden_features, hidden_features) for _ in range(hidden_layers)]
        layers += [nn.Linear(hidden_features, out_features)]
        self.net = nn.Sequential(*layers)

    def forward(self, coords):
        return torch.sigmoid(self.net(coords))

def build_spatial_grid(height, width):
    yy, xx = np.meshgrid(
        np.linspace(-1, 1, height, dtype=np.float32),
        np.linspace(-1, 1, width, dtype=np.float32),
        indexing="ij",
    )
    coords = np.stack([xx.ravel(), yy.ravel()], axis=1)
    return torch.from_numpy(coords).float()

def train_siren_from_mosaic(mosaic, masks, hidden_features=256, hidden_layers=3, steps=1000, learning_rate=1e-4):
    height, width = mosaic.shape
    coords = build_spatial_grid(height, width).to(DEVICE)
    mosaic_tensor = torch.from_numpy(mosaic[..., None]).float().to(DEVICE)
    masks_tensor = torch.from_numpy(masks).float().to(DEVICE)
    model = Siren(hidden_features, hidden_layers, out_features=3).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history = []

    model.train()
    for step in range(steps):
        rgb_prediction = model(coords).reshape(height, width, 3)
        mosaic_prediction = torch.sum(rgb_prediction * masks_tensor, dim=-1, keepdim=True)
        loss = F.mse_loss(mosaic_prediction, mosaic_tensor)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(loss.item())

    model.eval()
    with torch.no_grad():
        reconstruction = model(coords).reshape(height, width, 3).cpu().numpy()
    return model, np.clip(reconstruction, 0.0, 1.0).astype(np.float32), history

model_inr, I_hat_inr, loss_history = train_siren_from_mosaic(
    mosaic, bayer, hidden_features=256, hidden_layers=3, steps=1000
)

plt.figure(figsize=(5, 3))
plt.plot(loss_history)
plt.yscale("log")
plt.title("Perdida de entrenamiento SIREN")
plt.xlabel("Iteracion")
plt.ylabel("MSE")
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb_reference)
axes[0].set_title("Referencia RGB")
axes[1].imshow(mosaic, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Mosaico Bayer")
axes[2].imshow(I_hat_inr)
axes[2].set_title("Reconstruccion INR-SIREN")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 10. Red neuronal convolucional

El segundo enfoque neuronal consiste en entrenar una red convolucional que reciba como entrada la medicion mosaico y produzca directamente una imagen RGB.

Una forma conveniente de representar la entrada es concatenar la medicion $Y$ con las mascaras Bayer:

$$
X_{\text{CNN}} = \left[ Y, M_R, M_G, M_B \right]
\in \mathbb{R}^{H \times W \times 4}.
$$

La red convolucional aprende una funcion

$$
g_\phi: \mathbb{R}^{H \times W \times 4} \rightarrow \mathbb{R}^{H \times W \times 3},
$$

tal que

$$
\hat{I}_{\text{CNN}} = g_\phi(X_{\text{CNN}}).
$$

La perdida de entrenamiento puede definirse como

$$
\mathcal{L}(\phi) = \frac{1}{3HW}\left\|g_\phi(X_{\text{CNN}}) - I_{\text{ref}}\right\|_2^2.
$$

En esta primera implementacion se entrenara la red con una imagen controlada. Esto permite comprobar el funcionamiento del modelo y comparar metodos. Para estudiar generalizacion, posteriormente se puede construir un conjunto de parches a partir de varias imagenes RGB, siguiendo la estrategia del notebook `cnnDemosaicing.ipynb`.


In [ ]:
# Red neuronal convolucional
def build_cnn_input(mosaic, masks):
    """Construye X_CNN = [Y, M_R, M_G, M_B] con formato NCHW."""
    input_image = np.concatenate([mosaic[..., None], masks], axis=-1)
    return torch.from_numpy(input_image.transpose(2, 0, 1)[None]).float()

class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=padding),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size, padding=padding),
        )

    def forward(self, x):
        return x + self.block(x)

class DemosaicCNN(nn.Module):
    def __init__(self, input_channels=4, hidden_channels=32, num_blocks=4):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Conv2d(input_channels, hidden_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.residual_layers = nn.Sequential(
            *[ResidualBlock(hidden_channels) for _ in range(num_blocks)]
        )
        self.output_layer = nn.Conv2d(hidden_channels, 3, 3, padding=1)

    def forward(self, x):
        features = self.input_layer(x)
        features = self.residual_layers(features)
        return torch.sigmoid(self.output_layer(features))

def sensor_consistency_loss(prediction, input_tensor):
    """Penaliza que la reconstruccion no reproduzca la medicion Bayer."""
    mosaic_reprojected = torch.sum(prediction * input_tensor[:, 1:4], dim=1, keepdim=True)
    return F.mse_loss(mosaic_reprojected, input_tensor[:, 0:1])

def train_cnn(model, input_tensor, target_tensor, steps=1000, learning_rate=1e-3, physics_weight=0.1):
    """Entrena la CNN sobre el experimento controlado actual."""
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    for step in range(steps):
        prediction = model(input_tensor)
        rgb_loss = F.mse_loss(prediction, target_tensor)
        physics_loss = sensor_consistency_loss(prediction, input_tensor)
        loss = rgb_loss + physics_weight * physics_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history

cnn_input = build_cnn_input(mosaic, bayer).to(DEVICE)
cnn_target = torch.from_numpy(rgb_reference.transpose(2, 0, 1)[None]).float().to(DEVICE)
model_cnn = DemosaicCNN(input_channels=4, hidden_channels=32, num_blocks=4).to(DEVICE)
cnn_loss_history = train_cnn(model_cnn, cnn_input, cnn_target, steps=1000)

model_cnn.eval()
with torch.no_grad():
    cnn_output = model_cnn(cnn_input)[0].cpu().numpy()
I_hat_cnn = cnn_output.transpose(1, 2, 0).astype(np.float32)

plt.figure(figsize=(5, 3))
plt.plot(cnn_loss_history)
plt.yscale("log")
plt.title("Perdida de entrenamiento CNN")
plt.xlabel("Iteracion")
plt.ylabel("MSE")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb_reference)
axes[0].set_title("Referencia RGB")
axes[1].imshow(mosaic, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Mosaico Bayer")
axes[2].imshow(I_hat_cnn)
axes[2].set_title("Reconstruccion CNN")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 10.1. Entrenamiento de la CNN con varias imagenes

El entrenamiento con una sola imagen permite validar el pipeline, pero no permite medir generalizacion. Para obtener un conjunto de entrenamiento se descargan las 24 imagenes Kodak utilizadas en `cnnDemosaicing.ipynb` y se dividen en parches de tamano fijo.

Las primeras 20 imagenes se usan para entrenar y las ultimas 4 se reservan para evaluar. La separacion se realiza por imagen completa antes de crear los parches, evitando que parches de una misma imagen aparezcan en ambos conjuntos.


In [ ]:
# Descarga del dataset Kodak usado como entrenamiento supervisado
KODAK_DIR = DATA_DIR / "kodak"
KODAK_DIR.mkdir(exist_ok=True)
KODAK_ZIP = KODAK_DIR / "kodak.zip"
KODAK_URL = "https://arizona.box.com/shared/static/5bk6qar4vgfizmhum8onreu6sp90bz2i"

if not list(KODAK_DIR.rglob("kodim*.png")):
    !wget -q -O "{KODAK_ZIP}" "{KODAK_URL}"
    !unzip -q -o "{KODAK_ZIP}" -d "{KODAK_DIR}"

kodak_paths = sorted(KODAK_DIR.rglob("kodim*.png"))
if len(kodak_paths) < 8:
    raise RuntimeError(f"Se encontraron {len(kodak_paths)} imagenes Kodak; revisa la descarga del ZIP.")

train_paths = kodak_paths[:-4]
test_paths = kodak_paths[-4:]
print(f"Imagenes de entrenamiento: {len(train_paths)}")
print(f"Imagenes de prueba: {len(test_paths)}")

class KodakPatchDataset(Dataset):
    def __init__(self, image_paths, patch_size=64):
        self.samples = []
        self.patch_size = patch_size
        for path in image_paths:
            image = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
            height, width = image.shape[:2]
            for top in range(0, height - patch_size + 1, patch_size):
                for left in range(0, width - patch_size + 1, patch_size):
                    self.samples.append(image[top:top + patch_size, left:left + patch_size])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        rgb = self.samples[index]
        masks = bayer_masks(self.patch_size, self.patch_size)
        mosaic_patch = make_bayer_mosaic(rgb, masks)
        input_patch = np.concatenate([mosaic_patch[..., None], masks], axis=-1)
        input_tensor = torch.from_numpy(input_patch.transpose(2, 0, 1)).float()
        target_tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()
        return input_tensor, target_tensor

train_dataset = KodakPatchDataset(train_paths, patch_size=64)
test_dataset = KodakPatchDataset(test_paths, patch_size=64)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
print(f"Parches de entrenamiento: {len(train_dataset)}")
print(f"Parches de prueba: {len(test_dataset)}")

def train_cnn_loader(model, loader, epochs=10, learning_rate=1e-3, physics_weight=0.1):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for inputs, targets in loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            predictions = model(inputs)
            rgb_loss = F.mse_loss(predictions, targets)
            physics_loss = sensor_consistency_loss(predictions, inputs)
            loss = rgb_loss + physics_weight * physics_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * inputs.size(0)
        epoch_loss /= len(loader.dataset)
        history.append(epoch_loss)
        print(f"Epoca {epoch + 1:02d}/{epochs}: MSE = {epoch_loss:.6f}")
    return history

def evaluate_cnn_loader(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for inputs, targets in loader:
            predictions = model(inputs.to(DEVICE))
            losses.append(F.mse_loss(predictions.cpu(), targets).item())
    return float(np.mean(losses))

model_cnn_kodak = DemosaicCNN(input_channels=4, hidden_channels=32, num_blocks=4).to(DEVICE)
kodak_loss_history = train_cnn_loader(model_cnn_kodak, train_loader, epochs=10)
kodak_test_mse = evaluate_cnn_loader(model_cnn_kodak, test_loader)
print(f"MSE promedio en imagenes de prueba: {kodak_test_mse:.6f}")

plt.figure(figsize=(5, 3))
plt.plot(kodak_loss_history, marker="o")
plt.xlabel("Epoca")
plt.ylabel("MSE de entrenamiento")
plt.title("CNN entrenada con parches Kodak")
plt.grid(alpha=0.25)
plt.show()


## 11. Variantes experimentales

La guia pide probar diferentes variaciones de arquitectura y entrenamiento. Por lo tanto, se propone comparar distintas configuraciones, por ejemplo:

| Metodo | Arquitectura | Parametros a variar |
|---|---|---|
| Clasico | Interpolacion / OpenCV | Patron Bayer, metodo de interpolacion |
| INR | SIREN / MLP implicito | Numero de capas, ancho, frecuencia $\omega_0$, learning rate |
| CNN | Red convolucional | Numero de capas, canales ocultos, kernel size, epocas |

Para cada configuracion se deben registrar las metricas obtenidas, el tiempo de entrenamiento y observaciones visuales relevantes.


In [ ]:
# Variantes de arquitectura y entrenamiento para la CNN Kodak
cnn_experiments = [
    {"nombre": "CNN pequena", "hidden_channels": 16, "num_blocks": 2, "epochs": 5},
    {"nombre": "CNN base", "hidden_channels": 32, "num_blocks": 4, "epochs": 5},
]

def run_cnn_experiment(config):
    start_time = time.perf_counter()
    model = DemosaicCNN(
        input_channels=4,
        hidden_channels=config["hidden_channels"],
        num_blocks=config["num_blocks"],
    ).to(DEVICE)
    history = train_cnn_loader(
        model, train_loader, epochs=config["epochs"], physics_weight=0.1
    )
    test_mse = evaluate_cnn_loader(model, test_loader)
    return {
        "metodo": config["nombre"],
        "canales": config["hidden_channels"],
        "bloques": config["num_blocks"],
        "epocas": config["epochs"],
        "MSE prueba": test_mse,
        "tiempo [s]": time.perf_counter() - start_time,
    }, history

variant_rows = []
variant_histories = {}
for config in cnn_experiments:
    row, history = run_cnn_experiment(config)
    variant_rows.append(row)
    variant_histories[config["nombre"]] = history

variants_df = pd.DataFrame(variant_rows).sort_values("MSE prueba")
variants_df.round(4)


## 12. Metricas de evaluacion

Para comparar las reconstrucciones se usaran metricas entre la imagen de referencia $I_{\text{ref}}$ y cada estimacion $\hat{I}$.

### Error cuadratico medio

$$
\operatorname{MSE}(I, \hat{I}) = \frac{1}{3HW}\sum_{i=1}^{H}\sum_{j=1}^{W}\sum_{c=1}^{3}
\left(I_c(i,j) - \hat{I}_c(i,j)\right)^2.
$$

### PSNR

$$
\operatorname{PSNR}(I, \hat{I}) = 10 \log_{10}\left(\frac{\operatorname{MAX}_I^2}{\operatorname{MSE}(I, \hat{I})}\right).
$$

Si las imagenes estan normalizadas en $[0,1]$, entonces $\operatorname{MAX}_I = 1$.

### SSIM

El indice SSIM mide similitud estructural considerando luminancia, contraste y estructura. Se usara como complemento al PSNR, ya que este ultimo no siempre refleja adecuadamente la calidad visual percibida.


In [ ]:
# Metricas de calidad
def compute_metrics(reference, estimate):
    reference = np.clip(reference.astype(np.float32), 0.0, 1.0)
    estimate = np.clip(estimate.astype(np.float32), 0.0, 1.0)
    mse = float(np.mean((reference - estimate) ** 2))
    psnr = float(peak_signal_noise_ratio(reference, estimate, data_range=1.0))
    ssim_value = float(ssim(reference, estimate, channel_axis=-1, data_range=1.0))
    return {"MSE": mse, "PSNR [dB]": psnr, "SSIM": ssim_value}

results_df = pd.DataFrame([
    {"metodo": "Bilineal", **compute_metrics(rgb_reference, I_hat_classic)},
    {"metodo": "INR-SIREN", **compute_metrics(rgb_reference, I_hat_inr)},
    {"metodo": "CNN", **compute_metrics(rgb_reference, I_hat_cnn)},
])
results_df.round(4)

def evaluate_loader_metrics(model, loader):
    rows = []
    model.eval()
    with torch.no_grad():
        for inputs, targets in loader:
            predictions = model(inputs.to(DEVICE)).cpu().numpy().transpose(0, 2, 3, 1)
            targets = targets.numpy().transpose(0, 2, 3, 1)
            for target, prediction in zip(targets, predictions):
                rows.append(compute_metrics(target, prediction))
    return pd.DataFrame(rows).mean(numeric_only=True).to_dict()

kodak_test_metrics = evaluate_loader_metrics(model_cnn_kodak, test_loader)
kodak_test_results = pd.DataFrame([{"metodo": "CNN Kodak - prueba", **kodak_test_metrics}])
kodak_test_results.round(4)

if real_capture is not None:
    real_results_df = pd.DataFrame([
        {"metodo": "Real bilineal", **compute_metrics(real_reference, real_classic)},
        {"metodo": "Real OpenCV", **compute_metrics(real_reference, real_opencv)},
    ])
    display(real_results_df.round(4))


## 13. Visualizacion de resultados

Ademas de las metricas, se debe realizar una comparacion visual. Una grilla minima deberia incluir:

1. Imagen RGB de referencia.
2. Medicion mosaico Bayer.
3. Reconstruccion clasica.
4. Reconstruccion con representacion implicita.
5. Reconstruccion con CNN.
6. Mapas de error absoluto, si son utiles.

El error absoluto por pixel puede definirse como

$$
E(i,j) = \frac{1}{3}\sum_{c=1}^{3}\left|I_c(i,j)-\hat{I}_c(i,j)\right|.
$$


In [ ]:
# Visualizacion comparativa
def show_comparison(reference, mosaic, reconstructions, error_scale=5):
    """Muestra reconstrucciones y mapas de error en una grilla comun."""
    methods = list(reconstructions.keys())
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    axes[0, 0].imshow(reference)
    axes[0, 0].set_title("Referencia RGB")
    axes[0, 1].imshow(mosaic, cmap="gray", vmin=0, vmax=1)
    axes[0, 1].set_title("Mosaico Bayer")
    axes[0, 2].axis("off")

    for ax, (method, reconstruction) in zip(axes[1], reconstructions.items()):
        ax.imshow(np.clip(reconstruction, 0, 1))
        ax.set_title(method)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4))
    if len(methods) == 1:
        axes = [axes]
    for ax, (method, reconstruction) in zip(axes, reconstructions.items()):
        error = np.mean(np.abs(reference - reconstruction), axis=-1)
        ax.imshow(np.clip(error * error_scale, 0, 1), cmap="magma", vmin=0, vmax=1)
        ax.set_title(f"Error absoluto: {method}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

reconstructions = {
    "Bilineal": I_hat_classic,
    "INR-SIREN": I_hat_inr,
    "CNN": I_hat_cnn,
}
show_comparison(rgb_reference, mosaic, reconstructions)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["MSE", "PSNR [dB]", "SSIM"]):
    axes_index = results_df[metric].plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518", "#54A24B" if len(results_df) > 2 else "#4C78A8"])
    ax.set_title(metric)
    ax.set_xticklabels(results_df["metodo"], rotation=20)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Ejemplo de generalizacion sobre un parche Kodak no usado en el entrenamiento.
if "model_cnn_kodak" in globals() and len(test_dataset) > 0:
    test_input, test_target = test_dataset[0]
    with torch.no_grad():
        test_prediction = model_cnn_kodak(test_input[None].to(DEVICE))[0].cpu().numpy()
    test_reference = test_target.numpy().transpose(1, 2, 0)
    test_mosaic = test_input[0].numpy()
    test_reconstruction = test_prediction.transpose(1, 2, 0)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(test_reference)
    axes[0].set_title("Referencia Kodak")
    axes[1].imshow(test_mosaic, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Mosaico Bayer")
    axes[2].imshow(test_reconstruction)
    axes[2].set_title("CNN en imagen no vista")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 14. Analisis

En esta seccion se deben discutir los resultados obtenidos. Algunas preguntas guia son:

1. Que metodo obtiene mejor PSNR y SSIM?
2. Que metodo preserva mejor los bordes y detalles finos?
3. Se observan artefactos de color o patron Bayer residual?
4. La red implicita converge de manera estable?
5. Como afecta aumentar capas, ancho o epocas de entrenamiento?
6. La CNN mejora al modificar la arquitectura?
7. Que diferencias existen entre optimizar contra una imagen RGB procesada por el telefono y una referencia ideal?


## 15. Conclusiones

En esta tarea se espera concluir sobre la efectividad de metodos neuronales para demosaicking RGB a partir de datos con patron Bayer. La comparacion debe considerar tanto metricas objetivas como calidad visual.

Las conclusiones finales deberian mencionar:

1. El metodo con mejor desempeno cuantitativo.
2. El metodo con mejor calidad visual percibida.
3. Las principales dificultades durante el entrenamiento.
4. Las ventajas y limitaciones de las representaciones implicitas.
5. Las ventajas y limitaciones de las redes convolucionales.
6. Posibles mejoras futuras.
